# Graviton SDK: Astrophysics & N-Body Supercomputing
This notebook demonstrates how to run astrophysical N-body simulations using **REBOUND** and **Mercury6** on Graviton's high-density CPU Turing nodes.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CodeZeroLabs/graviton-examples/blob/master/graviton_sdk_astrophys_example.ipynb)

In [ ]:
# 1. Install the Graviton SDK
!pip install graviton-sdk

In [ ]:
import graviton as gv
import tempfile
import os

# 2. Authenticate with Graviton
gv.login("a-little-more-art-than-science")

## 1. Running a REBOUND Simulation
REBOUND simulations use a unified JSON structure specifying simulation duration, step resolution, and orbital bodies.

In [ ]:
rebound_payload = {
    "t": "rebound",
    "sim_time": 365.25,
    "time_step": 0.1,
    "bodies": [
        {"name": "Sun", "mass": 1.0, "x": 0.0, "y": 0.0, "z": 0.0},
        {"name": "Earth", "mass": 3.0e-6, "a": 1.0, "e": 0.0167},
        {"name": "Mars", "mass": 3.2e-7, "a": 1.524, "e": 0.0934}
    ]
}

# Evaluate complexity and estimated credits
eval_data = gv.evaluate(rebound_payload)
print(f"Job ID: {eval_data.id}")
print(f"Estimated Credit Cost: {eval_data.cost:.2f}")
print(f"Estimated Duration: {eval_data.estimated_time:.2f}s")

# Trigger computation on Turing swarm
job = gv.compute(eval_data.id)
print(f"Job status: {job.status()}")

## 2. Scraping and Executing Legacy Mercury6 Datasets
The Graviton SDK includes `gv.load_mercury(directory)` to parse legacy Fortran `param.in`, `big.in`, and `small.in` parameter files without manual conversion.

In [ ]:
# Create sample Mercury directory for demonstration
sample_dir = tempfile.mkdtemp(prefix="mercury_demo_")

param_content = """
)O+_06 Integration parameters
 start time (days) = 0.0
 stop time (days) = 365.25
 timestep (days) = 1.0
"""

big_content = """
)O+_06 Big bodies
 style = Cartesian
 Central_Star m=1.0 r=0.005 d=1.0
 0.0 0.0 0.0
 0.0 0.0 0.0
 0.0 0.0 0.0
 Planet_A m=0.001 r=0.001 d=1.0
 1.0 0.0 0.0
 0.0 6.28 0.0
 0.0 0.0 0.0
"""

with open(os.path.join(sample_dir, "param.in"), "w") as f:
    f.write(param_content)

with open(os.path.join(sample_dir, "big.in"), "w") as f:
    f.write(big_content)

# Automatically load and transform the dataset
mercury_payload = gv.load_mercury(sample_dir)
print("Transformed Payload:")
print(mercury_payload)

# Evaluate and compute
mercury_eval = gv.evaluate(mercury_payload)
print(f"Mercury Job ID: {mercury_eval.id}")
print(f"Cost: {mercury_eval.cost:.2f} credits")

mercury_job = gv.compute(mercury_eval.id)
print(f"Mercury Job provisioned: {mercury_job.job_id}")